# ST554 Final Project: Siona Benjamin
For this final project we will use Spark to handle streaming data and fitting a machine learning model. The data set in question describes power consumption from different zones of Tetoauan City in relation to factors such as time of day, temperature, and humidity. Our goal is to create a model that can predict the power consumption from a particular zone based off other variables in the dataset. This is beneficial if the measurement for a certain zone goes offline and needs to be determined another way. Once we create our model, we also want to make predictions about the power consumption in this zone in real time as new data is recieved. For this, we can take advantage of Spark's streaming capabilities. 

To get started, we of course need to import the necessary modules. Then, we'll read in our data as a pandas dataframe before converting this to a spark dataframe.

In [14]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder, CrossValidatorModel
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.regression import LinearRegression
from pyspark.sql.types import StructType
from pyspark.sql.functions import col
from pyspark.ml.feature import SQLTransformer, PCA, Binarizer, OneHotEncoder, VectorAssembler, StringIndexer

In [15]:
#create spark session
spark = SparkSession.builder.appName("final_project").getOrCreate()

In [16]:
#import data as pandas dataframe
power_data = pd.read_csv('power_ml_data.csv')
#convert pandas dataframe to spark dataframe
power_df = spark.createDataFrame(power_data)

Using `.show()` we can see what our data columns look like while `.dtypes` lets us see what data type each column is stored as. We see that most values are stored as doubles except for the Month and Hour variables.

In [17]:
power_df.show(10)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 18442.40964|    1|   0|
|      5.853|    76.9|     0.081|       

In [18]:
power_df.dtypes

[('Temperature', 'double'),
 ('Humidity', 'double'),
 ('Wind_Speed', 'double'),
 ('General_Diffuse_Flows', 'double'),
 ('Diffuse_Flows', 'double'),
 ('Power_Zone_1', 'double'),
 ('Power_Zone_2', 'double'),
 ('Power_Zone_3', 'double'),
 ('Month', 'bigint'),
 ('Hour', 'bigint')]

## Fitting the Model
The first part of this project will be training an elastic net model with out dataset to predict power consumption values for Zone 3. In an elastic net model, L1 (LASSO) and L2 (Ridge) penalties are combined to improve model predictions and stability. 

Now that we've loaded our dataset and have a good idea of what our data looks like, we can set up the transformations we want to apply to our data before training our model. The first transformation we'll apply is a SQL transformation to cast the Hour variable as a double instead of an integer. Within the same transformation, we will also rename the Power_Zone_3 column as label. After changing the Hour variable type, we'll apply a binarizer transformation to this variable to distinguish between night and day using 6.5 as the cutoff. Next , we'll use one-hot encoding to encode the Month variable. Additionally, we will run a PCA (Principle Component Analysis) fit on a few of the columns in our dataset. The PCA entails using a VectorAssembler transformation to place the desired variables together in a column followed by using the PCA transformation.

Lastly, we will use the VectorAssembler transformation to combine our desired predictor variables in a features column. 

In [6]:
#SQL transformer to cast Hour variable as DoubleType
sqlTrans = SQLTransformer(
    statement = """
                SELECT *,
                CAST(Hour AS DOUBLE) AS hour_double,
                Power_Zone_3 as label 
                FROM __THIS__
                """)

In [7]:
#Binarize transformer to convert continuous Hour values to binary values 
binarizer = Binarizer(threshold=6.5, inputCol="hour_double", outputCol="hour_binary")

In [8]:
#One-hot encoder to transform Month values to vector values
##StringIndexer transformation to conver Month values 
indexer = StringIndexer(inputCol="Month", outputCol="month_index")
##OneHotEncoder transformation
encoder = OneHotEncoder(inputCols=["Month"], outputCols=["month_vec"])

In [9]:
#PCA transformation 
##VectorAssembler to combine desired columns
pca_assembler = VectorAssembler(inputCols=["Temperature","Humidity","Wind_Speed","General_Diffuse_Flows","Diffuse_Flows"], outputCol="pca_features")
##PCA transformer 
pca = PCA(k=2,inputCol="pca_features", outputCol="pca_results")

In [10]:
#VectorAssembler to put predictors in features column 
assembler_features = VectorAssembler(inputCols=["hour_binary","Power_Zone_1","Power_Zone_2","month_vec","pca_results"], outputCol="features")

Now that we have defined our transformations, we can define the other components of our model. First we'll create an object to define our linear regression model. Then we'll define our parameter grid to set test values of `regParam`, which controls the amount of regularization, and `elasticNetParam`, which defines the balance between L1 and L2 regularization. We will also set up a pipeline with the transformations defined above and our linear regression model. 

In [11]:
#define object for linear regression model 
lr = LinearRegression()
#define parameter grid 
paramGrid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .addGrid(lr.elasticNetParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .build()
#define transformation pipeline 
pipeline = Pipeline(stages = [sqlTrans, binarizer, indexer, encoder, pca_assembler, pca, assembler_features, lr])

The next step is to set up our `CrossValidator` object and enter in our pipeline, parameter grid, and RMSE regression evaluator. For our cross validation, we'll use 5 folds. Now we can fit our cross validation model.

In [12]:
#set up cross validation 
crossval = CrossValidator(estimator = pipeline,
                          estimatorParamMaps = paramGrid,
                          evaluator = RegressionEvaluator(metricName='rmse'),
                          numFolds=5)

In [13]:
cvModel = crossval.fit(power_df)

26/04/28 17:10:07 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/04/28 17:10:07 WARN Instrumentation: [b6cdef7a] regParam is zero, which might cause numerical instability and overfitting.
26/04/28 17:10:09 WARN Instrumentation: [b6cdef7a] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/28 17:10:11 WARN Instrumentation: [148b699c] regParam is zero, which might cause numerical instability and overfitting.
26/04/28 17:10:12 WARN Instrumentation: [148b699c] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/28 17:10:14 WARN Instrumentation: [198ef797] regParam is zero, which might cause numerical instability and overfitting.
26/04/28 17:10:14 WARN Instrumentation: [198ef797] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
2

At this point, we've successfully used cross validation to train an elastic net model! Let's save this model to avoid having to rerun the training every time we reopen our notebook. We can use `.save()` to save our model in a folder, and `.load()` to reload this model when we want to.

In [14]:
#save model in directory
cvModel.write().overwrite().save("cvModel")

In [19]:
#load model from directory
cvModel = CrossValidatorModel.load("cvModel")

Let's see how our elastic net model performs. After cross validation, the optimal hyperparameters chosen were a `regParam` of 0.05 and an `elasticNetParam` of 0.1. 

In [61]:
#extract last training stage of the best model 
best_model = cvModel.bestModel.stages[-1]
#iterate through paramters in parameter map for the best model
print("Optimal Paramters:")
print("-" * 30)
for param, value in best_model.extractParamMap().items():
    #print regParam and elasticNetParam
    if param.name in [p.name for p in paramGrid[0].keys()]:
        print(f"{param.name}: {value}")

Optimal Paramters:
------------------------------
elasticNetParam: 0.1
regParam: 0.05


We can also take a look at the CV errors for each combination of `regParam` and `elasticNetParam`. We can see that many of the RMSE values eneded up being similar varying only in their decimal point values.

In [39]:
print("CV Errors:")
print("-" * 30)
for params, score in zip(paramGrid, cvModel.avgMetrics):
    param_str = "|".join([f"{param.name}={value}" for param, value in params.items()])
    print(f"{param_str} -- RMSE: {score:.4f}")

CV Errors:
------------------------------
regParam=0.0|elasticNetParam=0.0 -- RMSE: 2147.8759
regParam=0.0|elasticNetParam=0.05 -- RMSE: 2147.8759
regParam=0.0|elasticNetParam=0.1 -- RMSE: 2147.8759
regParam=0.0|elasticNetParam=0.25 -- RMSE: 2147.8759
regParam=0.0|elasticNetParam=0.5 -- RMSE: 2147.8759
regParam=0.0|elasticNetParam=0.75 -- RMSE: 2147.8759
regParam=0.0|elasticNetParam=0.9 -- RMSE: 2147.8759
regParam=0.0|elasticNetParam=0.95 -- RMSE: 2147.8759
regParam=0.0|elasticNetParam=0.98 -- RMSE: 2147.8759
regParam=0.0|elasticNetParam=0.99 -- RMSE: 2147.8759
regParam=0.0|elasticNetParam=1.0 -- RMSE: 2147.8759
regParam=0.05|elasticNetParam=0.0 -- RMSE: 2147.8758
regParam=0.05|elasticNetParam=0.05 -- RMSE: 2147.8768
regParam=0.05|elasticNetParam=0.1 -- RMSE: 2147.8751
regParam=0.05|elasticNetParam=0.25 -- RMSE: 2147.8762
regParam=0.05|elasticNetParam=0.5 -- RMSE: 2147.8753
regParam=0.05|elasticNetParam=0.75 -- RMSE: 2147.8756
regParam=0.05|elasticNetParam=0.9 -- RMSE: 2147.8755
regPar

Now we also want to calculate the resulting RMSE when we use our cvModel to predict Power_Zone_3 values of our original dataset. Doing so gives us an RMSE value of 2147.097. 

In [40]:
lr_rmse = RegressionEvaluator().evaluate(cvModel.transform(power_df))
print(f"RMSE: {lr_rmse}")

RMSE: 2147.0973169293934


Our last step with our model will be using it to add a residual column to our dataframe. First, we use our cvModel as a transformation to add a column of predicted values to our dataframe. We also create a column with residuals values showing the deviation of our predicted values from the original values. 

In [15]:
power_df_pred = cvModel.transform(power_df)
power_df_resid = power_df_pred.withColumn("residual",col("label")-col("prediction"))
power_df_resid.select("label","prediction","residual").show(8)

+-----------+------------------+------------------+
|      label|        prediction|          residual|
+-----------+------------------+------------------+
|20240.96386|20878.850660788765| -637.886800788765|
|20131.08434|18660.227266544003| 1470.857073455998|
|19668.43373| 18204.75215311452|1463.6815768854794|
|18899.27711|17590.648498339124|1308.6286116608753|
|18442.40964| 16997.30198645687|1445.1076535431312|
|18130.12048| 16517.68672349429|1612.4337565057103|
|17945.06024|16093.246141053492|1851.8140989465064|
|17459.27711|15722.695360253929|1736.5817497460703|
+-----------+------------------+------------------+
only showing top 8 rows


Now that we have successfully trained our elastic net model, we can move onto making predictions with new data.

## Streaming Data
In the previous section we trained an elastic net model on our dataset to predict the power consumption of Zone 3. With this model, we can now make predictions with new data that we read in from a stream. First we define a schema for the data that will be streamed in, following the schema from our `power_df` dataframe we used in the previous section. Next, we set up our stream using `.readStream()` and tell the stream to look for new data in the folder streaming_files. 

In [20]:
#define schema for read in data
myschema = power_df.schema

In [21]:
#read csv files from folder 'streaming_files' following schema 
stream_df = spark.readStream.schema(myschema).format("csv").option("header","true").load("streaming_files")

We also want to set up some tranformations to process our read in data. First, we'll rename the Power_Zone_3 column to label. Then we'll use our previously trained cvModel to predict values for Power_Zone_3 in our new data. At the same time, we'll also calculate the residuals for predicted values and select only the prediction, residual, and label columns from this dataframe. Once we've defined the transformations we want to do on our stream, we can use `.join()` to combine these dataframes and print them together to the console. Then, we'll use `.writeStream()` to write our transformed data stream to the console.

In [22]:
#transformation to rename response variable to label
rename_df = stream_df.withColumnRenamed("Power_Zone_3","label") 

#transformation to predict Power_Zone_3, create residual, and select columns
predict_df = cvModel.transform(stream_df).withColumn("residual",col("label")-col("prediction")).select("prediction", "residual","label")

#join rename_df and predict_df
joined_df=rename_df.join(predict_df, on='label')

#write stream to console
writeDF = joined_df.writeStream.outputMode("append").format("console").start()

26/04/29 22:46:25 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-723fb677-6da9-47d0-a70c-5a96d68c9658. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/29 22:46:25 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


At this point we have started our data stream and have definined which transormations we want to make to our data before it is written to the console. In a python file named `produce_stream_data.py`, we have a script that samples 5 records from a larger database and saves them in a csv file for us to read using our stream. Now that we have our stream read and write set up, we can run `produce_stream_data.py` to populate our streaming_files folder with csv files that will be read by the stream.

Once we've read in all of our data, we can stop writing to the console.

In [23]:
%run produce_stream_data.py

-------------------------------------------
Batch: 0
-------------------------------------------
+-----------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+------------------+-------------------+
|      label|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|        prediction|           residual|
+-----------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+------------------+-------------------+
|15172.82412|      13.47|   56.68|     0.084|                556.7|        590.1| 33693.55932| 21136.77812|    2|  11|16329.659614272965|-1156.8354942729657|
|13379.20327|      22.04|   65.99|     0.278|                0.062|        0.133| 27793.27434| 17060.70686|    9|   2| 13016.31812328516|  362.8851467148397|
|    20096.0|       20.5|   51.64|     0.091|                499.3|        242.9| 33766.37244| 19781.67006|    4|

-------------------------------------------
Batch: 1
-------------------------------------------
+-----------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+------------------+-------------------+
|      label|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|        prediction|           residual|
+-----------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+------------------+-------------------+
|17250.90909|      18.57|   56.84|     0.075|                662.5|        53.91| 34411.19483| 20463.54379|    4|  10| 19329.00602327414|-2078.0969332741406|
|14261.70468|      14.66|   67.12|     0.085|                0.077|        0.126| 31397.71863| 25531.75821|   12|  23|12498.729407577051| 1762.9752724229493|
|26962.70769|      21.46|    78.0|     0.065|                69.41|         67.6| 43524.23841| 26891.47609|    6|

-------------------------------------------
Batch: 2
-------------------------------------------
+-----------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+------------------+-------------------+
|      label|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|        prediction|           residual|
+-----------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+------------------+-------------------+
|11820.72289|      13.62|    82.9|     0.078|                0.062|          0.1| 24412.30769| 16214.87603|   11|   2|11030.350656073973|  790.3722339260257|
|16615.38462|      17.82|    82.4|     0.071|                0.062|        0.093| 28718.16393| 14507.73994|    5|   1|16991.578236259284|-376.19361625928286|
|17757.09091|       23.5|   41.31|     0.087|                884.0|        55.77| 34336.79225|  22138.9002|    4|

-------------------------------------------
Batch: 3
-------------------------------------------
+-----------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+------------------+-------------------+
|      label|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|        prediction|           residual|
+-----------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+------------------+-------------------+
|13691.03166|      25.96|   69.62|     4.923|                653.5|         57.5|  35898.0531| 22636.59044|    9|  14|15175.587056432076|-1484.5553964320752|
|13910.83417|      14.89|    71.0|     0.081|                0.051|        0.174| 22618.98305| 13579.33131|    2|   3| 13473.26613651243|  437.5680334875706|
|12728.02432|      23.42|    60.2|     0.088|                602.1|        61.09| 36797.19912| 23851.86722|   10|

-------------------------------------------
Batch: 4
-------------------------------------------
+-----------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+------------------+-------------------+
|      label|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|        prediction|           residual|
+-----------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+------------------+-------------------+
|16886.74699|        8.3|    78.6|     0.087|                0.037|        0.152| 26782.78481| 17482.06687|    1|   0|16620.413366111905| 266.33362388809473|
|15062.83417|      13.95|   56.17|     4.919|                0.084|        0.119| 23717.28814| 14097.26444|    2|   2|14193.408179099915|  869.4259909000848|
|25896.86747|      13.52|   57.93|     0.086|                6.976|         7.28| 41437.97468| 24685.71429|    1|

...

-------------------------------------------
Batch: 11
-------------------------------------------
+-----------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+------------------+-------------------+
|      label|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|        prediction|           residual|
+-----------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+------------------+-------------------+
|10245.37815|      16.71|   47.83|     0.087|                125.4|        138.6| 30515.58935| 25649.58576|   12|  16| 11305.85597354989|-1060.4778235498889|
|16105.64226|      14.54|   67.99|     0.087|                0.051|        0.115| 39415.96958|  33639.7668|   12|  19|18227.667285229963| -2122.025025229963|
|27972.92308|      22.13|   58.81|      4.92|                0.088|        0.096| 46550.46358| 27909.35551|    6

-------------------------------------------
Batch: 12
-------------------------------------------
+-----------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+-----------------+------------------+
|      label|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|       prediction|          residual|
+-----------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+-----------------+------------------+
| 18702.6506|      20.08|    79.7|     0.066|                0.102|        0.041| 36135.38462| 30804.54545|   11|  21| 17942.3158483663| 760.3347516337017|
|20676.92308|      24.38|   66.61|     4.923|                905.0|        57.95| 33847.94702| 17337.62994|    6|  12|17860.31376909396| 2816.609310906042|
|18411.53605|      22.81|    90.6|     4.917|                 0.08|        0.122| 29132.43063| 17654.06547|    8|   4|2143

-------------------------------------------
Batch: 13
-------------------------------------------
+-----------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+------------------+-------------------+
|      label|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|        prediction|           residual|
+-----------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+------------------+-------------------+
|25863.87692|      20.85|    82.9|      0.07|                0.051|        0.141| 38908.60927| 23931.39293|    6|   1|25535.903156731096|   327.973763268903|
|15885.59755|      24.98|   44.61|     0.272|                237.4|        53.66| 36643.53982| 22617.87942|    9|  18| 16543.25643293779| -657.6588829377888|
|18692.05312|       23.7|   62.15|     0.273|                 0.08|          0.1| 35018.76106| 21057.38046|    9

-------------------------------------------
Batch: 14
-------------------------------------------
+-----------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+------------------+-------------------+
|      label|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|        prediction|           residual|
+-----------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+------------------+-------------------+
|16256.38554|      16.14|    76.4|      0.07|                236.7|        241.2| 32943.79747| 20837.68997|    1|  13|17747.229546905255|-1490.8440069052558|
|10750.44534|      19.44|    77.9|      0.08|                 91.5|         74.1| 21038.16393| 11732.50774|    5|   7| 9871.782058307494|  878.6632816925066|
|24263.09623|      29.38|   38.13|      4.92|                849.0|        120.3| 33635.08306|  22572.1519|    7

-------------------------------------------
Batch: 15
-------------------------------------------
+-----------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+------------------+-------------------+
|      label|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|        prediction|           residual|
+-----------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+------------------+-------------------+
|25351.22257|      21.05|    74.6|     0.071|                0.073|        0.122| 33268.63485| 23508.34213|    8|   1|24571.380768408657|  779.8418015913448|
|18317.73279|      28.21|    35.6|     0.084|                763.0|        260.2|  36272.2623| 21815.47988|    5|  11| 18181.15884582646| 136.57394417353862|
|18480.97166|       22.0|   68.57|     4.919|                870.0|        170.4| 35554.62295| 21678.01858|    5

In [13]:
#stop stream once files have been read
writeDF.stop()

26/04/29 22:17:34 WARN DAGScheduler: Failed to cancel job group 76984595-671b-4487-92b6-467971e7ff4c. Cannot find active jobs for it.
26/04/29 22:17:35 WARN DAGScheduler: Failed to cancel job group 76984595-671b-4487-92b6-467971e7ff4c. Cannot find active jobs for it.


## Conclusion
In this project, we trained an elastic net model with pyspark MLlib to predict the power consumption for a certain zone in Tetoauan City from the other columns in the dataset such as temperature, wind speed, and measurements for other zones. Using cross validation resulted in an elastic net model with a regularization of 0.05 and an elastic net paramter of 0.1. Once we trained our model, we were then able to apply the model as a tranformation to make predictions with new data. We did this by setting up a stream to read in new data from our streaming_files folder. We also applied other transformation to the streamed in data, such as calculating the residual and relabeling column names to allow the joining of two separate dataframes. 

With this work, we see how to train a machine learned model with pyspark MLlib. We also saw how to set up a stream to efficiently read in and make predictions with new data.